In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql.window import Window

##Load silver orders

In [0]:
df_silver = spark.table(SILVER_ORDERS).select("payment_method").distinct()
print(f"Silver rows: {df_silver.count()}")
display(df_silver)

##Build dim_customer

In [0]:
df_dim_payment_method = (df_silver
    .filter(F.col("payment_method").isNotNull())
    .filter(F.col("payment_method") != "")

    # Surrogate key 
    .withColumn("payment_method_id",
        F.concat(
            F.lit("PAMA1"),
            F.lpad(
                F.row_number().over(
                    Window.partitionBy(F.lit(1)).orderBy("payment_method")
                ).cast("string"),
                5, "0"
            )
        )
    )

    .withColumn("_gold_ingested_at",
        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
    )

    .select(
        "payment_method_id",
        "payment_method",
        "_gold_ingested_at"
    )
)

print(f"dim_payment_method rows: {df_dim_payment_method.count()}")
display(df_dim_payment_method)

##Write to gold

In [0]:
(df_dim_payment_method
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_PAYMENT_METHOD)
)

print(f"✅ {df_dim_payment_method.count()} rows written to {GOLD_PAYMENT_METHOD}")

##Validate

In [0]:
df_check = spark.table(GOLD_PAYMENT_METHOD)

print(f"Total rows   : {df_check.count()}")
print(f"Distinct IDs : {df_check.select('payment_method_id').distinct().count()}")
df_check.show(5, truncate=False)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.gold.dim_payment_method;